# 📊 Project FORESIGHT – Notebook 03

# Feature Engineering

## Objective

This notebook transforms the integrated sales dataset into a machine learning-ready dataset by engineering predictive features.

The engineered features capture historical demand behaviour, calendar effects, pricing information, and temporal trends while preventing data leakage.

The final output of this notebook is used for demand forecasting and inventory risk prediction.
# 🔧 Safe Feature Engineering

This notebook creates features that can be engineered independently for each data partition.

These features do not depend on previous observations and therefore avoid any risk of temporal leakage while remaining memory efficient.

## Import Required Libraries

The following libraries are used for:

- Data manipulation using Pandas
- Numerical computations using NumPy
- File management using pathlib
- Memory management using the garbage collector

In [1]:
# ============================================================
# Import Libraries
# ============================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## Configure Project Directories

The project uses a structured folder hierarchy.

This section defines the locations of:

- Integrated data chunks
- Engineered feature chunks
- Processed data directory

Using relative paths keeps the notebook portable across different systems.

In [14]:
# ============================================================
# Project Paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

FINAL_CHUNK_DIR = PROCESSED_DATA_DIR / "final_chunks"

FEATURE_DIR = PROCESSED_DATA_DIR / "feature_chunks"

# Create folder if it doesn't exist
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

chunk_files = sorted(FINAL_CHUNK_DIR.glob("final_chunk_*.parquet"))

print("Final Chunk Directory :", FINAL_CHUNK_DIR)
print("Feature Directory     :", FEATURE_DIR)
print("Chunks Found          :", len(chunk_files))

Final Chunk Directory : c:\Users\Dell\OneDrive\Project_Foresight\data\processed\final_chunks
Feature Directory     : c:\Users\Dell\OneDrive\Project_Foresight\data\processed\feature_chunks
Chunks Found          : 64


## Memory-Efficient Feature Engineering

The complete dataset contains over 900,000 records.

Processing the entire dataset simultaneously would require a large amount of RAM.

To ensure scalability and reproducibility, the dataset is processed one chunk at a time.

For each chunk, the notebook:

1. Loads the data
2. Creates new features
3. Saves the engineered chunk
4. Releases memory before processing the next chunk

This approach follows production ETL best practices.

### Calendar Features

Calendar variables help the forecasting model capture recurring demand patterns.

The following temporal features are created:

- Day of month
- Week number
- Month
- Quarter
- Day of year
- Week of year
- Month start indicator
- Month end indicator
- Weekend indicator

These features improve the model's ability to learn seasonal behaviour.

### Pricing Features

Retail demand is strongly influenced by selling price.

This notebook engineers two pricing variables:

- **has_price** – Indicates whether pricing information is available.
- **log_price** – Log-transformed selling price used to reduce skewness and stabilize variance.

Log transformation is commonly used in machine learning because it reduces the influence of extreme values.

### Event Features

Special events and SNAP programmes often affect customer purchasing behaviour.

The following binary indicators are created:

- **has_event** – Whether a special event occurs on a given day.
- **has_snap** – Whether SNAP benefits are active in any state.

These variables help the forecasting model learn demand spikes caused by external events.

In [15]:
# ============================================================
# Safe Feature Engineering
# ============================================================

for i, file in enumerate(chunk_files, start=1):

    print(f"Processing chunk {i}/{len(chunk_files)}")

    chunk = pd.read_parquet(file)

    # -------------------------
    # Date Features
    # -------------------------

    chunk["date"] = pd.to_datetime(chunk["date"])

    chunk["day"] = chunk["date"].dt.day.astype("int8")

    chunk["week"] = chunk["date"].dt.isocalendar().week.astype("int16")

    chunk["month"] = chunk["date"].dt.month.astype("int8")

    chunk["quarter"] = chunk["date"].dt.quarter.astype("int8")

    chunk["year"] = chunk["date"].dt.year.astype("int16")

    chunk["day_of_year"] = chunk["date"].dt.dayofyear.astype("int16")

    chunk["week_of_year"] = chunk["date"].dt.isocalendar().week.astype("int16")

    chunk["is_month_start"] = (
        chunk["date"].dt.is_month_start
    ).astype("int8")

    chunk["is_month_end"] = (
        chunk["date"].dt.is_month_end
    ).astype("int8")

    chunk["is_weekend"] = (
        chunk["weekday"]
        .isin(["Saturday", "Sunday"])
    ).astype("int8")

    # -------------------------
    # Price Features
    # -------------------------

    chunk["has_price"] = (
        chunk["sell_price"].notna()
    ).astype("int8")

    chunk["log_price"] = np.log1p(
        chunk["sell_price"].fillna(0)
    )

    # -------------------------
    # Event Features
    # -------------------------

    chunk["has_event"] = (
        chunk["event_name_1"].notna()
    ).astype("int8")

    chunk["has_snap"] = (
        (
            chunk["snap_CA"]
            + chunk["snap_TX"]
            + chunk["snap_WI"]
        ) > 0
    ).astype("int8")

    # -------------------------
    # Save
    # -------------------------

    chunk.to_parquet(
        FEATURE_DIR / file.name,
        index=False
    )

    del chunk
    gc.collect()

print("\n✅ Safe Feature Engineering Complete")

Processing chunk 1/64
Processing chunk 2/64
Processing chunk 3/64
Processing chunk 4/64
Processing chunk 5/64
Processing chunk 6/64
Processing chunk 7/64
Processing chunk 8/64
Processing chunk 9/64
Processing chunk 10/64
Processing chunk 11/64
Processing chunk 12/64
Processing chunk 13/64
Processing chunk 14/64
Processing chunk 15/64
Processing chunk 16/64
Processing chunk 17/64
Processing chunk 18/64
Processing chunk 19/64
Processing chunk 20/64
Processing chunk 21/64
Processing chunk 22/64
Processing chunk 23/64
Processing chunk 24/64
Processing chunk 25/64
Processing chunk 26/64
Processing chunk 27/64
Processing chunk 28/64
Processing chunk 29/64
Processing chunk 30/64
Processing chunk 31/64
Processing chunk 32/64
Processing chunk 33/64
Processing chunk 34/64
Processing chunk 35/64
Processing chunk 36/64
Processing chunk 37/64
Processing chunk 38/64
Processing chunk 39/64
Processing chunk 40/64
Processing chunk 41/64
Processing chunk 42/64
Processing chunk 43/64
Processing chunk 44/

### Save Engineered Chunks

After feature engineering, each processed chunk is saved to disk.

Saving chunks individually provides several advantages:

- Lower memory usage
- Faster recovery if processing stops
- Easier debugging
- Scalability for large datasets

The saved chunks will be used directly in the forecasting pipeline.

## Feature Verification

A sample engineered chunk is loaded to verify that all newly created features have been generated successfully.

This quality check ensures the feature engineering pipeline completed correctly before model development begins.

In [16]:
feature_files = sorted(
    FEATURE_DIR.glob("*.parquet")
)

sample = pd.read_parquet(feature_files[0])

print(sample.shape)

sample[
    [
        "date",
        "day",
        "week",
        "month",
        "quarter",
        "day_of_year",
        "week_of_year",
        "is_weekend",
        "has_price",
        "log_price",
        "has_event",
        "has_snap",
    ]
].head()

(914700, 34)


,date,day,week,month,quarter,day_of_year,week_of_year,is_weekend,has_price,log_price,has_event,has_snap
0,2011-01-29,29,4,1,1,29,4,1,0,0.0,0,0
1,2011-01-29,29,4,1,1,29,4,1,0,0.0,0,0
2,2011-01-29,29,4,1,1,29,4,1,0,0.0,0,0
3,2011-01-29,29,4,1,1,29,4,1,0,0.0,0,0
4,2011-01-29,29,4,1,1,29,4,1,0,0.0,0,0


# ✅ Notebook Summary

Feature engineering has been successfully completed.

### Features Engineered

✔ Calendar Features

- Day
- Week
- Month
- Quarter
- Day of Year
- Week of Year
- Month Start
- Month End
- Weekend Indicator

✔ Pricing Features

- Price Availability
- Log Selling Price

✔ Event Features

- Event Indicator
- SNAP Indicator

---

### Outcomes

- Memory-efficient chunk processing
- Machine learning-ready feature dataset
- No manual preprocessing
- Reproducible feature engineering pipeline

The engineered dataset produced in this notebook serves as the input for Notebook 04, where time-series forecasting features, baseline models, and machine learning models will be developed.